# Nâng cấp model 1D-CNN phân loại ECG — vòng 4 (giảm phương sai, chốt cấu hình)
**Kết luận vòng 3:** độ lệch giữa các seed (Se S ±0,18; Se F ±0,23) **lớn hơn chênh lệch giữa các cấu hình**, nên chưa thể kết luận cấu hình nào tốt hơn một cách tin cậy. Đặc trưng độ không đều RR (E7) không giúp nhịp N ở bản ghi rung nhĩ và làm mất hẳn lớp F.

**Vòng 4 tập trung vào độ ổn định để chốt model đưa lên phần cứng:**
- **EMA trọng số** (trung bình trượt trọng số trong lúc train): giảm dao động giữa các lần train, **không tốn phần cứng** vì kết quả vẫn là một bộ trọng số.
- **5 seed** cho mỗi cấu hình, learning rate thấp hơn (5e-4), batch lớn hơn.
- **Chuẩn hóa RR theo từng bệnh nhân**: tỉ số RR so với trung vị ~300 nhịp trước (~5 phút, nhân quả), thay cho đặc trưng độ không đều.
- Chỉ so sánh 3 ứng viên chốt:

| Mã | Nội dung |
|---|---|
| F1 | Thuần hình dạng, cửa sổ dài, không RR (E6r + EMA) |
| F2 | Nhánh RR 8 đặc trưng + co giãn thời gian (E8 bỏ độ không đều, + EMA) |
| F3 | F2 + 2 tỉ số RR so với trung vị 300 nhịp (chuẩn hóa theo bệnh nhân) |

Model cuối được chọn theo **cvF1**; nếu chênh lệch cvF1 nhỏ hơn độ lệch chuẩn DS2, **ưu tiên model đơn giản hơn** (ít phần cứng hơn).

**Chạy:** Runtime → Change runtime type → **GPU**, rồi Runtime → Run all.

In [2]:
!pip install -q wfdb
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Cấu hình chung

In [3]:
import os, json, time
import numpy as np

DATA_DIR = '/content/drive/MyDrive/mitdb_extracted/mit-bih-arrhythmia-database-1.0.0'   # thư mục chứa 100.dat, 100.hea, 100.atr ...
OUT_DIR  = '/content/drive/MyDrive/ecg_processed/improve_v4'                 # nơi lưu kết quả, model
os.makedirs(OUT_DIR, exist_ok=True)

FS_RAW = 360
SEED   = 42

# 4 lớp theo khuyến nghị AAMI EC57 / de Chazal (lớp Q gần như không có trong DS1/DS2
# vì các bản ghi máy tạo nhịp 102, 104, 107, 217 bị loại). Đặt USE_Q = True để giữ 5 lớp.
USE_Q = False

DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203,
       205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214,
       219, 221, 222, 228, 231, 232, 233, 234]

AAMI = {'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
        'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
        'V': 'V', 'E': 'V',
        'F': 'F',
        '/': 'Q', 'f': 'Q', 'Q': 'Q'}
BEAT_SYMBOLS = set(AAMI.keys())
CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q'] if USE_Q else ['N', 'S', 'V', 'F']
NUM_CLASSES = len(CLASS_NAMES)
print('Số lớp:', NUM_CLASSES, CLASS_NAMES)

Số lớp: 4 ['N', 'S', 'V', 'F']


## 2. Tiền xử lý: lọc nhiễu, cắt nhịp, đặc trưng RR
- **Lọc**: loại trôi đường nền bằng 2 bộ lọc trung vị (200 ms, 600 ms) + thông thấp Butterworth 40 Hz (chạy trên CPU/PS, không tốn FPGA).
- **Cửa sổ**: tính theo giây nên đổi tần số lấy mẫu không làm lệch vị trí đỉnh R.
- **Chuẩn hóa từng nhịp**: trừ trung vị, chia giá trị tuyệt đối lớn nhất → khoảng [-1, 1] (khớp đầu vào 8-bit frac 6).
- **RR (10 số)**: RR trước, RR sau, RR trung bình 10 nhịp trước (nhân quả), 5 tỉ số so với nền 10/30 nhịp, 2 tỉ số so với **trung vị 300 nhịp** (chuẩn hóa theo bệnh nhân). Model chọn số đặc trưng dùng qua tham số `n_rr` (8 hoặc 10).

In [4]:
import wfdb
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt

def load_record(rec):
    path = os.path.join(DATA_DIR, str(rec))
    r = wfdb.rdrecord(path)
    ch = r.sig_name.index('MLII') if 'MLII' in r.sig_name else 0
    sig = r.p_signal[:, ch].astype(np.float64)
    ann = wfdb.rdann(path, 'atr')
    return sig, np.asarray(ann.sample), np.asarray(ann.symbol)

def clean_signal(sig, fs=FS_RAW):
    base = median_filter(sig, size=int(0.2 * fs) | 1, mode='nearest')
    base = median_filter(base, size=int(0.6 * fs) | 1, mode='nearest')
    x = sig - base
    b, a = butter(4, 40.0 / (fs / 2), btype='low')
    return filtfilt(b, a, x)

def extract_record(rec, pre_s, post_s, ds, filt):
    sig, samples, symbols = load_record(rec)
    if filt:
        sig = clean_signal(sig)
    fs = FS_RAW / ds
    if ds > 1:
        sig = sig[::ds]                        # đã lọc thông thấp 40 Hz nên hạ mẫu an toàn
    pre, post = int(round(pre_s * fs)), int(round(post_s * fs))

    is_beat = np.array([s in BEAT_SYMBOLS for s in symbols])
    r_all = samples[is_beat] / ds
    sym_all = symbols[is_beat]
    rr = np.diff(r_all) / fs                   # giây

    X, R, Y = [], [], []
    for i in range(1, len(r_all) - 1):
        lab = AAMI[sym_all[i]]
        if lab == 'Q' and not USE_Q:
            continue
        c = int(round(r_all[i]))
        if c - pre < 0 or c + post > len(sig):
            continue
        w = sig[c - pre:c + post]
        w = w - np.median(w)
        m = np.max(np.abs(w))
        if m < 1e-9:
            continue
        pre_rr, post_rr = rr[i - 1], rr[i]
        local = np.mean(rr[max(0, i - 10):i])
        local30 = np.mean(rr[max(0, i - 30):i])
        med300 = np.median(rr[max(0, i - 300):i])
        X.append((w / m).astype(np.float32))
        R.append([pre_rr, post_rr, local, pre_rr / local, post_rr / local, pre_rr / post_rr,
                  pre_rr / local30, post_rr / local30, pre_rr / med300, post_rr / med300])
        Y.append(CLASS_NAMES.index(lab))
    return np.array(X), np.array(R, dtype=np.float32), np.array(Y), np.full(len(Y), rec)

_cache = {}
def build_dataset(records, pre_s, post_s, ds, filt):
    key = (tuple(records), pre_s, post_s, ds, filt)
    if key not in _cache:
        parts = [extract_record(r, pre_s, post_s, ds, filt) for r in records]
        _cache[key] = tuple(np.concatenate([p[k] for p in parts]) for k in range(4))
    return _cache[key]

def describe(y, name):
    cnt = np.bincount(y, minlength=NUM_CLASSES)
    print(name, dict(zip(CLASS_NAMES, cnt.tolist())), 'tổng', len(y))

## 3. Model thân thiện phần cứng
Cấu trúc giữ đúng các khối đã có IP HLS. Nhánh RR (nếu bật) ghép vào sau Flatten, trước Dense 1 — phần cứng chỉ thêm 6 thanh ghi đầu vào AXI-Lite. Đầu ra là **logit** (softmax bỏ đi vì phần cứng dùng argmax).

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

N_RR = 10

def build_model(win_len, conv_blocks, use_rr, dense_units=32, rr_dense=0):
    beat_in = layers.Input(shape=(win_len, 1), name='beat')
    x = beat_in
    for i, (f, k) in enumerate(conv_blocks, 1):
        x = layers.Conv1D(f, k, padding='same', name=f'conv1d_{i}')(x)
        x = layers.ReLU(name=f'relu_{i}')(x)
        x = layers.MaxPooling1D(2, name=f'pool_{i}')(x)
    x = layers.Flatten(name='flatten')(x)
    inputs = [beat_in]
    if use_rr:
        rr_in = layers.Input(shape=(CUR_NRR,), name='rr')
        inputs.append(rr_in)
        r = layers.Dense(rr_dense, activation='relu', name='rr_dense')(rr_in) if rr_dense else rr_in
        x = layers.Concatenate(name='concat')([x, r])
    x = layers.Dense(dense_units, activation='relu', name='dense_1')(x)
    out = layers.Dense(NUM_CLASSES, name='dense_out')(x)   # logit
    return models.Model(inputs, out)

CUR_NRR = 10   # số đặc trưng RR dùng cho thí nghiệm hiện tại (đặt trong vòng lặp)

def rr_normalize(R):
    """Đưa RR về khoảng ~[0, 2]: giây chia 1.5, tỉ số chia 2. Cố định, không học -> dễ làm phần cứng."""
    R = R[:, :CUR_NRR]
    return (R / np.array([1.5, 1.5, 1.5, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0], dtype=np.float32)[:CUR_NRR]).clip(0, 3.9)

## 4. Huấn luyện: trọng số lớp, focal loss, tăng cường dữ liệu

In [6]:
def class_weights_effective(y, beta=0.999, max_ratio=10.0):
    cnt = np.maximum(np.bincount(y, minlength=NUM_CLASSES), 1)
    w = (1 - beta) / (1 - np.power(beta, cnt))
    w = w / w.min()
    w = np.minimum(w, max_ratio)
    return w / np.mean(w[np.bincount(y, minlength=NUM_CLASSES) > 0])

def focal_loss(alpha, gamma=2.0):
    alpha = tf.constant(alpha, dtype=tf.float32)
    def loss(y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        logp = tf.nn.log_softmax(logits)
        idx = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
        lp = tf.gather_nd(logp, idx)
        p = tf.exp(lp)
        a = tf.gather(alpha, y_true)
        return tf.reduce_mean(-a * tf.pow(1.0 - p, gamma) * lp)
    return loss

def time_stretch(x, max_frac=0.10):
    """Co giãn cả batch theo 1 hệ số ngẫu nhiên, giữ tâm (đỉnh R) cố định."""
    L = tf.shape(x)[1]
    s = tf.random.uniform([], 1.0 - max_frac, 1.0 + max_frac)
    newL = tf.cast(tf.round(tf.cast(L, tf.float32) * s), tf.int32)
    y = tf.image.resize(x[:, :, None, :], [newL, 1], method='bilinear')[:, :, 0, :]
    diff = newL - L
    y = tf.cond(diff > 0,
                lambda: y[:, diff // 2: diff // 2 + L, :],
                lambda: tf.pad(y, [[0, 0], [(-diff) // 2, -diff - (-diff) // 2], [0, 0]], mode='SYMMETRIC'))
    return tf.reshape(y, tf.shape(x))

def augment_beats(x, stretch=False):
    """x: (B, L, 1). Biên độ, nhiễu, trôi nền, dịch thời gian nhỏ (+ co giãn)."""
    if stretch:
        x = time_stretch(x)
    B = tf.shape(x)[0]; L = tf.shape(x)[1]
    scale = tf.random.uniform([B, 1, 1], 0.85, 1.15)
    noise = tf.random.normal(tf.shape(x), stddev=0.02)
    t = tf.cast(tf.range(L), tf.float32)[None, :, None] / tf.cast(L, tf.float32)
    wander = tf.random.uniform([B, 1, 1], -0.08, 0.08) * tf.sin(
        2.0 * np.pi * tf.random.uniform([B, 1, 1], 0.2, 1.0) * t + tf.random.uniform([B, 1, 1], 0, 6.28))
    shift = tf.random.uniform([], -4, 5, dtype=tf.int32)
    x = tf.roll(x, shift, axis=1)
    return tf.clip_by_value(x * scale + noise + wander, -1.98, 1.98)

def make_ds(Xb, R, y, use_rr, batch, training, stretch=False):
    feats = {'beat': Xb[..., None]}
    if use_rr:
        feats['rr'] = R
    ds = tf.data.Dataset.from_tensor_slices((feats, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED)
    ds = ds.batch(batch)
    if training:
        def aug(f, yy):
            f = dict(f); f['beat'] = augment_beats(f['beat'], stretch); return f, yy
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

## 5. Đánh giá chuẩn AAMI + mô phỏng 8-bit + hiệu chỉnh bias

In [7]:
from sklearn.metrics import confusion_matrix

def aami_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    res = {}
    f1s, ses = [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp
        se = tp / (tp + fn) if tp + fn else np.nan
        pp = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * se * pp / (se + pp) if (se + pp) and not np.isnan(se) else 0.0
        res[CLASS_NAMES[c]] = dict(Se=se, PP=pp, F1=f1, n=int(cm[c].sum()))
        if cm[c].sum():
            f1s.append(f1); ses.append(se)
    res['acc'] = np.trace(cm) / cm.sum()
    res['macroF1'] = float(np.mean(f1s))
    res['balAcc'] = float(np.mean(ses))
    res['cm'] = cm
    return res

def print_metrics(m, title):
    print(f'--- {title} ---  acc={m["acc"]:.4f}  macroF1={m["macroF1"]:.4f}  balAcc={m["balAcc"]:.4f}')
    for c in CLASS_NAMES:
        d = m[c]
        if d['n']:
            print(f'   {c}: Se={d["Se"]:.3f}  +P={d["PP"]:.3f}  F1={d["F1"]:.3f}  (n={d["n"]})')
    print('   Confusion (hàng=thật, cột=đoán):'); print(m['cm'])

# ---------- Lượng tử hóa kiểu phần cứng (Q-format 8-bit, thang lũy thừa 2) ----------
def frac_bits_for(x, bits=8):
    m = float(np.max(np.abs(x)))
    if m < 1e-12:
        return bits - 1
    return int(max(bits - 1 - max(int(np.ceil(np.log2(m))), 0), 0))

def fq(x, f, bits=8):
    s = 2.0 ** f
    return np.clip(np.round(x * s), -2 ** (bits - 1), 2 ** (bits - 1) - 1) / s

def _conv_same(x, W, b):
    """x (n,L,C), W keras (k,C,F)."""
    k = W.shape[0]; pl = (k - 1) // 2; pr = k - 1 - pl
    xp = np.pad(x, ((0, 0), (pl, pr), (0, 0)))
    win = np.lib.stride_tricks.sliding_window_view(xp, k, axis=1)     # (n,L,C,k)
    return np.einsum('nlck,kcf->nlf', win, W, optimize=True) + b

def forward_np(model, Xb, R, conv_blocks, use_rr, qcfg=None, batch=2048, rr_dense=0):
    """Suy luận NumPy từng tầng; qcfg=None -> float, ngược lại fake-quant 8-bit như phần cứng."""
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    Q = (lambda x, key: fq(x, qcfg[key])) if qcfg else (lambda x, key: x)
    outs = []
    for s in range(0, len(Xb), batch):
        x = Q(Xb[s:s + batch, :, None].astype(np.float64), 'input')
        for i in range(1, len(conv_blocks) + 1):
            W, b = Wd[f'conv1d_{i}']
            if qcfg:
                W, b = fq(W, qcfg[f'conv1d_{i}.kernel']), fq(b, qcfg[f'conv1d_{i}.bias'])
            x = Q(np.maximum(_conv_same(x, W, b), 0), f'relu_{i}')
            Lp = x.shape[1] // 2
            x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
        x = x.reshape(len(x), -1)
        if use_rr:
            r = Q(rr_normalize(R[s:s + batch]).astype(np.float64), 'rr')
            if rr_dense:
                W, b = Wd['rr_dense']
                if qcfg:
                    W, b = fq(W, qcfg['rr_dense.kernel']), fq(b, qcfg['rr_dense.bias'])
                r = Q(np.maximum(r @ W + b, 0), 'rr_dense')
            x = np.concatenate([x, r], axis=1)
        W, b = Wd['dense_1']
        if qcfg:
            W, b = fq(W, qcfg['dense_1.kernel']), fq(b, qcfg['dense_1.bias'])
        x = Q(np.maximum(x @ W + b, 0), 'dense_1')
        W, b = Wd['dense_out']
        if qcfg:
            W, b = fq(W, qcfg['dense_out.kernel']), fq(b, qcfg['dense_out.bias'])
        outs.append(x @ W + b)                    # logit giữ độ chính xác đầy đủ như phần cứng
    return np.concatenate(outs)

def calibrate(model, Xb, R, conv_blocks, use_rr, n=3000, rr_dense=0):
    """Chọn frac_bits cho trọng số + activation, hiệu chuẩn trên dữ liệu TRAIN."""
    idx = np.random.default_rng(SEED).choice(len(Xb), min(n, len(Xb)), replace=False)
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    q = {'input': 6}
    for name, (W, b) in Wd.items():
        q[f'{name}.kernel'], q[f'{name}.bias'] = frac_bits_for(W), frac_bits_for(b)
    x = Xb[idx, :, None].astype(np.float64)
    for i in range(1, len(conv_blocks) + 1):
        W, b = Wd[f'conv1d_{i}']
        x = np.maximum(_conv_same(x, W, b), 0); q[f'relu_{i}'] = frac_bits_for(x)
        Lp = x.shape[1] // 2; x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
    x = x.reshape(len(x), -1)
    if use_rr:
        rr = rr_normalize(R[idx]); q['rr'] = frac_bits_for(rr)
        if rr_dense:
            W, b = Wd['rr_dense']; rr = np.maximum(rr @ W + b, 0); q['rr_dense'] = frac_bits_for(rr)
        x = np.concatenate([x, rr], axis=1)
    W, b = Wd['dense_1']; q['dense_1'] = frac_bits_for(np.maximum(x @ W + b, 0))
    return q

def tune_bias(logits, y, iters=3, grid=np.linspace(-3, 3, 61)):
    """Tìm độ lệch cộng vào logit từng lớp để tối đa macro-F1 trên VALIDATION."""
    off = np.zeros(NUM_CLASSES)
    best = aami_metrics(y, np.argmax(logits, 1))['macroF1']
    for _ in range(iters):
        present = set(np.unique(y).tolist())
        for c in [c for c in range(1, NUM_CLASSES) if c in present]:   # giữ N làm mốc; bỏ lớp vắng
            for g in grid:
                o = off.copy(); o[c] = g
                f = aami_metrics(y, np.argmax(logits + o, 1))['macroF1']
                if f > best + 1e-6:
                    best, off = f, o
    return off, best

## 6. Danh sách thí nghiệm
Bỏ comment / sửa để thử cấu hình khác. `ds` = hệ số hạ mẫu (1: 360 Hz, 2: 180 Hz).

In [8]:
LONG = dict(pre_s=0.9, post_s=1.1, ds=2, filt=True, conv=[(16, 7), (32, 5), (32, 5)], aug=True, loss='focal')
EXPERIMENTS = [
    dict(name='F1_noRR',          **LONG, use_rr=False, n_rr=0,  rr_dense=0,  cw_cap=10, stretch=False),
    dict(name='F2_rr8_str',       **LONG, use_rr=True,  n_rr=8,  rr_dense=16, cw_cap=25, stretch=True),
    dict(name='F3_rr10_med_str',  **LONG, use_rr=True,  n_rr=10, rr_dense=16, cw_cap=25, stretch=True),
]
SEEDS = [1, 2, 3, 4, 5]
LR = 5e-4
MAX_EPOCHS, BATCH, CV_FOLDS, PATIENCE = 50, 512, 3, 10

## 7. Chạy thí nghiệm: k-fold theo bệnh nhân → train cuối trên toàn DS1 → đánh giá DS2 một lần
- Bản ghi **208** (nguồn nhịp F duy nhất của DS1) luôn nằm trong tập train; các bản ghi còn lại chia vào `CV_FOLDS` nhóm, cân bằng số nhịp S và V giữa các nhóm.
- Mỗi fold: chọn epoch có **macro-F1 validation** cao nhất và tìm độ lệch bias trên validation.
- Model cuối: train trên toàn DS1 với số epoch = trung vị các fold, bias = trung bình các fold.

In [9]:
tf.keras.utils.set_random_seed(SEED)

def make_folds(recs_y, k):
    """recs_y: dict rec -> nhãn. Chia bản ghi (trừ 208) thành k nhóm cân bằng S, V."""
    order = sorted([r for r in recs_y if r != 208],
                   key=lambda r: -(np.sum(recs_y[r] == 1) * 5 + np.sum(recs_y[r] == 2)))
    folds, load = [[] for _ in range(k)], np.zeros((k, 2))
    for r in order:
        j = int(np.argmin(load[:, 0] * 5 + load[:, 1]))
        folds[j].append(r)
        load[j] += [np.sum(recs_y[r] == 1), np.sum(recs_y[r] == 2)]
    return folds

class MacroF1Checkpoint(tf.keras.callbacks.Callback):
    def __init__(self, val_inputs, yv, patience):
        super().__init__(); self.vi, self.yv, self.pat = val_inputs, yv, patience
    def on_train_begin(self, logs=None):
        self.best, self.best_ep, self.wait, self.best_w = -1, 0, 0, None
    def on_epoch_end(self, epoch, logs=None):
        lg = self.model.predict(self.vi, batch_size=2048, verbose=0)
        f = aami_metrics(self.yv, lg.argmax(1))['macroF1']
        print(f'   epoch {epoch + 1:2d}  val macroF1 = {f:.4f}')
        if f > self.best:
            self.best, self.best_ep, self.wait, self.best_w = f, epoch + 1, 0, self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.pat:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)

def train_one(E, Xa, Ra, ya, epochs, val=None, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = build_model(Xa.shape[1], E['conv'], E['use_rr'], rr_dense=E['rr_dense'])
    cw = class_weights_effective(ya, max_ratio=E['cw_cap'])
    loss = focal_loss(cw) if E['loss'] == 'focal' else \
        tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(tf.keras.optimizers.Adam(LR, use_ema=True, ema_momentum=0.99), loss=loss)
    ds_tr = make_ds(Xa, rr_normalize(Ra), ya, E['use_rr'], BATCH, training=E['aug'], stretch=E['stretch'])
    fit_kw = {} if E['loss'] == 'focal' else {'class_weight': dict(enumerate(cw))}
    cbs = []
    if val is not None:
        Xv, Rv, yv = val
        vi = [Xv[..., None]] + ([rr_normalize(Rv)] if E['use_rr'] else [])
        ck = MacroF1Checkpoint(vi, yv, PATIENCE); cbs.append(ck)
    model.fit(ds_tr, epochs=epochs, verbose=0, callbacks=cbs, **fit_kw)
    return model, (ck.best_ep, ck.best) if val is not None else (epochs, None)

results, per_rec_rows = [], []
for E in EXPERIMENTS:
    t0 = time.time()
    CUR_NRR = max(E['n_rr'], 1)
    print('\n' + '=' * 70 + f'\n{E["name"]}\n' + '=' * 70)
    X1, R1, y1, g1 = build_dataset(DS1, E['pre_s'], E['post_s'], E['ds'], E['filt'])
    Xte, Rte, yte, gte = build_dataset(DS2, E['pre_s'], E['post_s'], E['ds'], E['filt'])
    describe(y1, 'DS1'); describe(yte, 'DS2')
    folds = make_folds({r: y1[g1 == r] for r in DS1}, CV_FOLDS)

    best_eps, offsets, cv_f1 = [], [], []
    for k, vrecs in enumerate(folds):
        vmask = np.isin(g1, vrecs)
        print(f'-- fold {k + 1}/{CV_FOLDS}: val = {vrecs}')
        describe(y1[vmask], '   val')
        model, (ep, f1v) = train_one(E, X1[~vmask], R1[~vmask], y1[~vmask], MAX_EPOCHS,
                                     val=(X1[vmask], R1[vmask], y1[vmask]))
        lg_v = forward_np(model, X1[vmask], R1[vmask], E['conv'], E['use_rr'], rr_dense=E['rr_dense'])
        off, f1b = tune_bias(lg_v, y1[vmask])
        print(f'   fold {k + 1}: epoch tốt nhất {ep}, val macroF1 {f1v:.4f} -> sau bias {f1b:.4f}, offset {np.round(off, 2)}')
        best_eps.append(ep); offsets.append(off); cv_f1.append(f1v)

    ep_final = int(np.median(best_eps)); off_final = np.mean(offsets, axis=0)
    print(f'>> Train cuối trên toàn DS1: {ep_final} epoch, offset {np.round(off_final, 2)}, seeds {SEEDS}')
    kw = dict(conv_blocks=E['conv'], use_rr=E['use_rr'], rr_dense=E['rr_dense'])
    no232 = gte != 232
    for seed in SEEDS:
        model, _ = train_one(E, X1, R1, y1, ep_final, seed=seed)
        qcfg = calibrate(model, X1, R1, **kw)
        lgq = forward_np(model, Xte, Rte, qcfg=qcfg, **kw) + off_final
        pred = lgq.argmax(1)
        m = aami_metrics(yte, pred)
        print_metrics(m, f'{E["name"]} seed {seed} | DS2 8-bit + bias')
        s_no232 = (pred[(yte == 1) & no232] == 1).mean()
        row = dict(exp=E['name'], seed=seed, params=model.count_params(), cvF1=np.mean(cv_f1),
                   acc=m['acc'], macroF1=m['macroF1'], balAcc=m['balAcc'], Se_S_no232=s_no232)
        for c in CLASS_NAMES:
            row[f'Se_{c}'] = m[c]['Se']; row[f'PP_{c}'] = m[c]['PP']; row[f'F1_{c}'] = m[c]['F1']
        results.append(row)
        for rec in DS2:
            mk = gte == rec
            pr = {'exp': E['name'], 'seed': seed, 'rec': rec}
            for ci, c in enumerate(CLASS_NAMES):
                sel = mk & (yte == ci)
                pr[f'n_{c}'] = int(sel.sum()); pr[f'Se_{c}'] = (pred[sel] == ci).mean() if sel.sum() else np.nan
            per_rec_rows.append(pr)
        model.save(os.path.join(OUT_DIR, f'{E["name"]}_s{seed}.keras'))
        with open(os.path.join(OUT_DIR, f'{E["name"]}_s{seed}_quant.json'), 'w') as f:
            json.dump(dict(qcfg=qcfg, offset=off_final.tolist(), epochs=ep_final, seed=seed, cfg=E), f, indent=1)
    print(f'CV macroF1 theo bệnh nhân: {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}')
    print(f'Thời gian: {(time.time() - t0) / 60:.1f} phút')


F1_noRR
DS1 {'N': 45814, 'S': 943, 'V': 3786, 'F': 414} tổng 50957
DS2 {'N': 44206, 'S': 1836, 'V': 3219, 'F': 388} tổng 49649
-- fold 1/3: val = [209, 119, 114, 108]
   val {'N': 7715, 'S': 399, 'V': 505, 'F': 6} tổng 8625
   epoch  1  val macroF1 = 0.4811
   epoch  2  val macroF1 = 0.4846
   epoch  3  val macroF1 = 0.4581
   epoch  4  val macroF1 = 0.4447
   epoch  5  val macroF1 = 0.4509
   epoch  6  val macroF1 = 0.4391
   epoch  7  val macroF1 = 0.4704
   epoch  8  val macroF1 = 0.4561
   epoch  9  val macroF1 = 0.4809
   epoch 10  val macroF1 = 0.4797
   epoch 11  val macroF1 = 0.5361
   epoch 12  val macroF1 = 0.5269
   epoch 13  val macroF1 = 0.5537
   epoch 14  val macroF1 = 0.5118
   epoch 15  val macroF1 = 0.5544
   epoch 16  val macroF1 = 0.5665
   epoch 17  val macroF1 = 0.5411
   epoch 18  val macroF1 = 0.5653
   epoch 19  val macroF1 = 0.5614
   epoch 20  val macroF1 = 0.5465
   epoch 21  val macroF1 = 0.5576
   epoch 22  val macroF1 = 0.5525
   epoch 23  val macroF1 = 

## 8. Bảng tổng hợp: trung bình ± độ lệch chuẩn theo seed (dùng cho bài báo)
Chọn cấu hình theo **cvF1** (validation theo bệnh nhân). Cột `Se_S_no232` chỉ để phân tích, không thay chỉ số chính.

In [10]:
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format); pd.set_option('display.width', 250)
df = pd.DataFrame(results)
df.to_csv(os.path.join(OUT_DIR, 'results_all_seeds.csv'), index=False)
cols = ['cvF1', 'acc', 'macroF1', 'balAcc', 'Se_S_no232'] + \
       [f'{k}_{c}' for c in CLASS_NAMES for k in ('Se', 'PP', 'F1')]
summary = df.groupby('exp')[cols].agg(['mean', 'std'])
display(summary)
summary.to_csv(os.path.join(OUT_DIR, 'results_summary.csv'))

pretty = df.groupby('exp')[cols].agg(lambda v: f'{v.mean():.3f} ± {v.std():.3f}')
display(pretty.T)

best = df.groupby('exp')['cvF1'].mean().idxmax()
print('Cấu hình được chọn theo cvF1:', best)

pr = pd.DataFrame(per_rec_rows)
pr.to_csv(os.path.join(OUT_DIR, 'per_record_all_seeds.csv'), index=False)
display(pr.groupby(['exp', 'rec']).mean(numeric_only=True).drop(columns='seed')
          .loc[best][['n_N', 'Se_N', 'n_S', 'Se_S', 'n_V', 'Se_V', 'n_F', 'Se_F']])
print('Đã lưu results_all_seeds.csv, results_summary.csv, per_record_all_seeds.csv vào', OUT_DIR)

cvF1         acc       macroF1       balAcc       Se_S_no232        ...  PP_V        F1_V        Se_F        PP_F        F1_F      
                 mean   std  mean   std    mean   std   mean   std       mean   std  ...  mean   std  mean   std  mean   std  mean   std  mean   std
exp                                                                                  ...                                                            
F1_noRR         0.480 0.000 0.926 0.005   0.439 0.023  0.407 0.026      0.150 0.063  ... 0.964 0.019 0.731 0.074 0.000 0.000 0.000 0.000 0.000 0.000
F2_rr8_str      0.513 0.000 0.939 0.002   0.569 0.012  0.550 0.011      0.602 0.023  ... 0.948 0.010 0.910 0.005 0.042 0.026 0.339 0.095 0.074 0.043
F3_rr10_med_str 0.513 0.000 0.929 0.004   0.527 0.010  0.513 0.013      0.673 0.025  ... 0.953 0.005 0.878 0.016 0.000 0.000 0.000 0.000 0.000 0.000

[3 rows x 34 columns]

exp,F1_noRR,F2_rr8_str,F3_rr10_med_str
cvF1,0.480 ± 0.000,0.513 ± 0.000,0.513 ± 0.000
acc,0.926 ± 0.005,0.939 ± 0.002,0.929 ± 0.004
macroF1,0.439 ± 0.023,0.569 ± 0.012,0.527 ± 0.010
balAcc,0.407 ± 0.026,0.550 ± 0.011,0.513 ± 0.013
Se_S_no232,0.150 ± 0.063,0.602 ± 0.023,0.673 ± 0.025
Se_N,0.995 ± 0.005,0.978 ± 0.004,0.973 ± 0.006
PP_N,0.929 ± 0.006,0.961 ± 0.002,0.954 ± 0.003
F1_N,0.961 ± 0.003,0.969 ± 0.001,0.964 ± 0.002
Se_S,0.037 ± 0.016,0.305 ± 0.040,0.266 ± 0.041
PP_S,0.266 ± 0.096,0.343 ± 0.016,0.271 ± 0.036


Cấu hình được chọn theo cvF1: F2_rr8_str


,n_N,Se_N,n_S,Se_S,n_V,Se_V,n_F,Se_F
rec,,,,,,,,
100,2236.000,1.000,33.000,0.909,1.000,1.000,0.000,NaN
103,2080.000,1.000,2.000,0.000,0.000,NaN,0.000,NaN
105,2524.000,0.977,0.000,NaN,41.000,0.512,0.000,NaN
111,2121.000,0.999,0.000,NaN,1.000,1.000,0.000,NaN
113,1786.000,0.997,6.000,1.000,0.000,NaN,0.000,NaN
117,1532.000,1.000,1.000,1.000,0.000,NaN,0.000,NaN
121,1859.000,1.000,1.000,0.000,1.000,1.000,0.000,NaN
123,1513.000,1.000,0.000,NaN,3.000,1.000,0.000,NaN
200,1741.000,0.993,30.000,0.647,825.000,0.923,2.000,0.000


Đã lưu results_all_seeds.csv, results_summary.csv, per_record_all_seeds.csv vào /content/drive/MyDrive/ecg_processed/improve_v4
